In [1]:
from huggingface_hub import login
from dotenv import load_dotenv
import os 

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

login(token=hf_token)

/home/bouchra/env_ia_master/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
import os
from mistralai import Mistral


api_key_mistral = os.getenv("API_KEY_MISTRAL")

# Connexion au client
client = Mistral(api_key=api_key_mistral)

model_name = "mistral-small-latest"



L'agent ReAct (Reasoning and Acting) représente une avancée significative dans le domaine des systèmes d'intelligence artificielle capables de raisonner et d'agir de manière autonome. Contrairement aux modèles de langage traditionnels qui se contentent de générer du texte, un agent ReAct combine la capacité de raisonnement structuré avec la possibilité d'exécuter des actions concrètes dans son environnement.

<img src="https://blent-courses.s3.eu-west-1.amazonaws.com/llm-engineer/img/react_diagram.svg" width="800" />

In [3]:
def llm_inference(prompt: str) -> str:
    """
    Réalise une inférence via l'API Mistral AI.
    Plus de GPU, plus de torch, juste une requête web.
    """
    try:
        response = client.chat.complete(
            model="mistral-small-latest", # On utilise Devstral-Small via API
            messages=[
                # On envoie tout le prompt formaté comme un message utilisateur.
                # Le modèle est assez intelligent pour suivre les instructions internes.
                {"role": "user", "content": prompt} 
            ],
            temperature=0.1, # Faible température pour être rigoureux sur le format
            max_tokens=1000  # On limite la longueur de la réponse
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erreur API : {e}"

Vérifions que notre inférence au LLM fonctionne bien.

In [4]:
llm_inference("Comment ça va ?")

"Ça va bien, merci ! 😊 Et toi, comment ça va aujourd'hui ? 🌟"

Outils

In [5]:
def ask_human(ask: str) -> str: 
    """Ask human to input a text.
    
    Parameters
    ----------
    ask: str
        The request asked to the user.
        
    Returns
    -------
    str
        The user answer.
    """
    return input(ask)

def upload(name: str) -> str:
    """Propose a user to upload a document/picture.

    Parameters
    ----------
    name: str
        The name/type of document expected.
        
    Returns
    -------
    str
        The name of the document.
    """
    
tools = {
    "AskHuman": ask_human,
    "Upload": upload
}

Créons une fonction qui va permettre de produire notre template de prompt.

In [ ]:
def format_prompt(history: list[str]) -> str:
    """Template de prompt pour l'agent ReAct."""
    prompt = """
Tu es un agent IA spécialiste de la déclaration de sinistres pour une assurance habitation. Tu utilises des étapes de réflexion et action pour répondre à la demande des assurés concernant leur déclaration de sinistres. 

## Contexte

Afin de pouvoir répondre à l'assuré, tu trouveras ci-dessous les éléments de contexte spécifiques aux processus de la déclaration de sinistres, et les garanties de l'assuré.

--------

{processus}

--------

## Actions

Voici les actions disponibles.

{actions}

Voici quelques exemples de réflexions et d'actions.

## Exemple

### Entrée

Question: J'ai eu un sinistre hier.

### Sortie

Thought: J'ai besoin de connaître le type de sinistre.
Action: AskHuman
Arguments:
  ask=Quel type de sinistre avez-vous eu ?

## Exemple

### Entrée

Question: Il y a eu une tempête ce matin avec des grêlons, et ma baie vitré à été impactée (ci-joint la photo).

### Sortie

Thought: J'ai toute les informations concernant le processus "3. Bris de glace".
Answer: Suite du processus. 

## Règles

- Si tu ne trouves pas d'outil approprié, ne fournis pas d'action et donne simplement la réponse.
- Lorsque tu fournis des arguments, tu dois les définir sous la forme `nom_argument=valeur_argument`, avec une ligne par argument.
- Tu ne dois pas ré-écrire dans la sortie, ce qu'il y a en entrée.
- Tu dois juste afficher la sortie qui est censé être la suite du raisonnement entrée : tu ne dois avoir qu'une seule paire réflexion/action, ou directement la réponse.
- Tu dois supposer que nous sommes en 2026 si l'utilisateur n'indique pas d'année spécifique.
- Tu dois t'assurer que conformément au processus lié au sinistre, toutes les informations ont bien été récoltées de la part de l'utilisateur.
- Évite de ré-utiliser des éléments du processus dans tes raisonnements : ces derniers doivent être succincts.
- Tu ne dois fournir la réponse finale qu'une fois tous les attendus de l'assuré pour le type de sinistre actuel (date du sinistre, documents, détails, etc) ont bien été renseignés.
- Ne demande pas plusieurs informations en une seule fois, mais fait-le étapes par étapes.

## Inférence

Propose maintenant une nouvelle réflexion et une nouvelle action et/ou réponse pour l'historique suivant.

### Entrée

{history}

### Sortie

""".strip()
    actions = "\n\n".join([
        f"### Action `{t}`\n\n{tools[t].__doc__}"
        for t in tools
    ])
    return prompt.format(
        actions=actions,
        processus=open("./processus.md", "r").read(),
        history="\n".join([x.strip() for x in history])
    )

Maintenant, nous créons deux fonctions :

- `parse_output` permet de **parser** la réponse du LLM, notamment de récupérer correctement les arguments ou la réponse finale.
- `react_agent` permet de créer la boucle réflexion/action de notre agent ReAct.

In [7]:
import re

def parse_output(output: str) -> tuple[str, ...]:
    """Permet de traiter la sortie ReAct d'un LLM.
    
    Args:
        output (str): La sortie brute du LLM.
    
    Returns:
        tuple[str, ...]: Un tuple dont le premier élément est le type de sortie ("thought", "action" ou "answer"), et les éléments suivants sont les contenus associés.
        - Si le type est "thought", le second élément est la réflexion.
        - Si le type est "action", le second élément est le nom de l'outil, et le troisième élément est un dictionnaire des arguments.
        - Si le type est "answer", le second élément est la réponse finale.
    """
    action_match = re.search(r"Action: (\w+)", output)  # On récupère le nom de l'outil
    arguments_match = re.search(r"Arguments:([\s\S]*)", output)  # On récupère les arguments
    answer_match = re.search(r"Answer: (.*)", output)

    if answer_match:  # Si l'agent propose une réponse, alors on l'utilise en priorité
        return "answer", answer_match.group(1).strip()
    elif action_match:
        tool_name = action_match.group(1)
        tool_input = {}
        if arguments_match is not None:  # Dans ce cas, on dispose au moins d'un argument
            tool_input = {}
            for arg in arguments_match.group(1).split("\n"):
                if len(arg.strip()) == 0:  # Si le texte est vide, on ignore
                    continue
                argument_re = re.search(r"^\s*(.*)\=(.*)$", arg)  # On récupère d'un part le nom de l'argument, et d'autre part sa valeur
                if argument_re is None or argument_re is not None and len(argument_re.groups()) != 2:
                    raise ValueError(f"Impossible to parse argument '{arg}' for action '{tool_name}'.")
                tool_input[argument_re.group(1)] = argument_re.group(2)  # Ensuite, on récupère l'argument et sa valeur (tout sera donc en str)
        return "action", tool_name, tool_input
    else:
        return "thought", output.strip()

def react_agent(question: str, max_turns: int = 10):
    """Implémente un agent ReAct simple.
    
    Args:
        question (str): La question à poser à l'agent.
        max_turns (int, optional): Le nombre maximum de tours de réflexion/action. Par défaut 10.
        
    Returns:
        str: L'historique complet de l'agent, y compris la réponse finale.
    """
    history = [f"Question: {question}"]
    for i in range(max_turns):
        prompt = format_prompt(history)
        output = llm_inference(prompt)
        print("-" * 20)
        print(f"Étape {i+1}")
        print("-" * 20)
        print(output)
        step_type, *content = parse_output(output)
        if step_type == "action":
            tool_name, tool_input = content
            if tool_name in tools:
                history.append(f"Thought: Je vais utiliser l'outil {tool_name}.")
                if tool_input:
                    observation = tools[tool_name](**tool_input)
                    history.append(f"Action: {tool_name}[{tool_input}]")
                else:
                    observation = tools[tool_name]()
                    history.append(f"Action: {tool_name}")
                history.append(f"Observation: {observation}")
            else:
                history.append(f"Observation: Error - Unknown tool {tool_name}")
        elif step_type == "answer":
            history.append(f"Answer: {content[0]}")
            return "\n".join(history)
        else:
            history.append(f"Thought: {content[0]}")
    return "\n".join(history) + "\nAnswer: Could not reach a conclusion."

Nous pouvons ensuite tester sur quelques exemples.

In [8]:
result = react_agent("je veux déclarer un sinistre")
# ma machine à laver s'est mal fermée et ça a tout inondé

--------------------
Étape 1
--------------------
Thought: Je dois d'abord identifier le type de sinistre pour suivre le processus approprié.
Action: AskHuman
Arguments:
  ask=Quel type de sinistre souhaitez-vous déclarer ? (Dégât des eaux, Bris de glace, Vol)
--------------------
Étape 2
--------------------
Thought: Je dois maintenant suivre le processus "3. Bris de glace" et commencer par demander la date du sinistre.
Action: AskHuman
Arguments:
  ask=Quelle est la date du sinistre ?
--------------------
Étape 3
--------------------
Thought: Je vais utiliser l'outil AskHuman pour demander les détails du bris de glace.
Action: AskHuman
Arguments:
  ask=Quelle est l'élément vitré endommagé et comment s'est-il cassé ?
--------------------
Étape 4
--------------------
Thought: Il me manque la preuve visuelle pour finaliser la déclaration.
Action: Upload
Arguments:
  name=photo_bris_de_glace
--------------------
Étape 5
--------------------
Thought: Toutes les informations et documents n

In [9]:
result = react_agent("bonjour, j'ai une fenêtre cassée à cause d'un orage survenu il y a 3 jours.")

--------------------
Étape 1
--------------------
Thought: Le sinistre déclaré est un "Bris de glace". Je dois suivre le processus correspondant et commencer par demander la date du sinistre.
Action: AskHuman
Arguments:
  ask=Pouvez-vous me confirmer la date exacte du sinistre ?
--------------------
Étape 2
--------------------
Thought: J'ai besoin de recueillir les détails du bris de glace (élément cassé et circonstances).
Action: AskHuman
Arguments:
  ask=Pouvez-vous me préciser quel élément vitré a été endommagé et comment cela s'est produit ?
--------------------
Étape 3
--------------------
Thought: Je vais demander à l'assuré de fournir une photo de l'élément vitré endommagé.
Action: Upload
Arguments:
  name=photo_bris_de_glace
--------------------
Étape 4
--------------------
Thought: Toutes les informations nécessaires pour le sinistre "Bris de glace" ont été collectées (date, détails, preuve visuelle).
Answer: Votre déclaration de sinistre pour le bris de glace est maintenant 